# FMA split diagnostics

The saved tables below show the two pre-audit splits generated on WCSS. They contain aggregate results. The analysis reads the shared catalogue on WCSS when rerun.

Set `VERSION` to the version printed by `atlas fma-split`. Set `COMPARE_VERSION` to another version built from the same inputs or to `None`. The tables compare the published FMA split, a seeded random assignment of groups and the group-stratified split. A pre-audit dataset uses the verified inventory and exact duplicates.

In [1]:
import json

import pandas as pd
from IPython.display import display

from music_atlas.settings import paths

VERSION = "pre-audit-v1"
COMPARE_VERSION = "pre-audit-rare-support-v1"
data = paths()["data"]
ready = data / "ready"
result = ready / VERSION
if not (result / "tracks.parquet").is_file():
    available = sorted(p.parent.name for p in ready.glob("*/tracks.parquet"))
    raise ValueError(f"Set VERSION to one of: {', '.join(available)}")
summary = json.loads((result / "summary.json").read_text(encoding="utf-8"))
support = pd.read_csv(result / "class_support.csv")
versions = [VERSION]
if COMPARE_VERSION and COMPARE_VERSION != VERSION:
    comparison_dir = ready / COMPARE_VERSION
    if (comparison_dir / "tracks.parquet").is_file():
        other_summary = json.loads((comparison_dir / "summary.json").read_text(encoding="utf-8"))
        if all(summary[key] == other_summary[key] for key in ("inputs", "schema", "eligible")):
            versions.append(COMPARE_VERSION)
        else:
            print(f"{COMPARE_VERSION}: choose a comparison version with the same inputs.")
    else:
        print(f"{COMPARE_VERSION}: comparison manifest unavailable.")
status = "Audited dataset" if summary.get("audio_audited") else "Pre-audit dataset"
print(f"{VERSION} | {status} | {summary['version']}")
display(pd.DataFrame(summary["splits"]).T)
display(pd.DataFrame(summary["overlap"]).T)

pre-audit-v1 | Pre-audit dataset | pre-audit-v1-8d68f7a1aa80ba0f


,train,validation,test
canonical,85260,10657,10657
group_random,85250,10673,10651
published,84353,10958,11263


,artists,duplicate_groups,components
canonical,0,0,0
group_random,0,0,0
published,0,11,5


## Label support and frequency baselines

The first table shows the largest shift in label prevalence in percentage points, relative to the selected annotated catalogue. The next tables count positive tracks and independent groups per class. The all-negative baseline can achieve high label accuracy by predicting the common absence of tags. Macro recall and macro-F1 give equal weight to each scored class and expose that failure.

In [2]:
display(support.groupby(["scheme", "split"])["prevalence_shift"].apply(
    lambda values: 100 * values.abs().max()
).round(2).rename("Max absolute shift (pp)").unstack())
canonical = support[support["scheme"].eq("canonical")]
display(canonical.pivot(index="genre", columns="split", values=["tracks", "groups"]))
for split, scores in summary["baselines"].items():
    print(f"{split}: constant-score mAP {scores['constant_score_map']:.3f}")
    display(pd.DataFrame({k: v for k, v in scores.items() if isinstance(v, dict)}).T.round(3))

split,test,train,validation
scheme,,,
canonical,0.20,0.04,0.13
group_random,3.85,0.92,4.04
published,1.59,0.52,5.02


tracks                   groups                 
split                 test  train validation   test train validation
genre                                                               
Blues                  171   1410        171     17   135         17
Classical              407   3292        407     45   361         45
Country                194   1594        199     28   218         27
Easy Listening          71    588         71     18   134         18
Electronic            3438  27537       3438    614  4902        614
Experimental          3812  30532       3810    695  5553        696
Folk                  1274  10163       1269    151  1210        151
Hip-Hop                839   6714        836    100   799        100
Instrumental          1490  11956       1492    229  1827        229
International          524   4223        524    105   829        105
Jazz                   409   3306        411     71   568         71
Old-Time / Historic     84    699         85     57   407         56
Pop                   1383  11078       1384    203  1627        203
Rock                  3293  26341       3289    457  3658        458
Soul-RnB               149   1203        147     27   211         27
Spoken                 189   1502        185     38   302         39

validation: constant-score mAP 0.107


,label_accuracy,macro_recall,macro_balanced_accuracy,macro_f1,micro_f1
all_negative,0.893,0.000,0.5,0.000,0.000
most_frequent_genre,0.877,0.062,0.5,0.034,0.271


test: constant-score mAP 0.107


,label_accuracy,macro_recall,macro_balanced_accuracy,macro_f1,micro_f1
all_negative,0.893,0.000,0.5,0.000,0.000
most_frequent_genre,0.877,0.062,0.5,0.034,0.271


## Proportional and rare-support splits

Compare held-out positive tracks and independent groups per genre. Both versions partition the same catalogue. Use the three files from one version together.

In [3]:
if len(versions) == 2:
    other_version = versions[1]
    other = pd.read_csv(ready / other_version / "class_support.csv")
    comparison = pd.concat([
        canonical.assign(version=VERSION),
        other[other["scheme"].eq("canonical")].assign(version=other_version),
    ])
    comparison = comparison[comparison["split"].isin(["validation", "test"])].copy()
    comparison["Tracks / groups"] = (
        comparison["tracks"].astype(str) + " / " + comparison["groups"].astype(str)
    )
    display(comparison.pivot(index="genre", columns=["version", "split"],
                             values="Tracks / groups"))

version             pre-audit-v1             pre-audit-rare-support-v1  \
split                 validation        test                validation   
genre                                                                    
Blues                   171 / 17    171 / 17                  260 / 24   
Classical               407 / 45    407 / 45                  407 / 45   
Country                 199 / 27    194 / 28                  253 / 34   
Easy Listening           71 / 18     71 / 18                  149 / 33   
Electronic            3438 / 614  3438 / 614                3436 / 614   
Experimental          3810 / 696  3812 / 695                3810 / 696   
Folk                  1269 / 151  1274 / 151                1269 / 151   
Hip-Hop                836 / 100   839 / 100                 835 / 100   
Instrumental          1492 / 229  1490 / 229                1497 / 228   
International          524 / 105   524 / 105                 524 / 105   
Jazz                    411 / 71    409 / 71                  408 / 72   
Old-Time / Historic      85 / 56     84 / 57                  178 / 96   
Pop                   1384 / 203  1383 / 203                1383 / 203   
Rock                  3289 / 458  3293 / 457                3292 / 457   
Soul-RnB                147 / 27    149 / 27                  253 / 44   
Spoken                  185 / 39    189 / 38                  250 / 50   

version                          
split                      test  
genre                            
Blues                  246 / 24  
Classical              407 / 45  
Country                249 / 34  
Easy Listening         149 / 33  
Electronic           3439 / 614  
Experimental         3812 / 695  
Folk                 1265 / 152  
Hip-Hop               835 / 100  
Instrumental         1494 / 228  
International         524 / 104  
Jazz                   411 / 71  
Old-Time / Historic    178 / 97  
Pop                  1383 / 203  
Rock                 3286 / 458  
Soul-RnB               248 / 44  
Spoken                 251 / 50

## Catalogue coverage and balanced evaluation

Coverage counts use the published manifest. Before the audio audit, duration is taken from FMA metadata and eligibility is provisional. Untagged tracks stay in the partition and are excluded from supervised genre evaluation.

The balanced diagnostic samples the same number of tagged and untagged-by-class tracks per scored genre in validation and test, with one track per independent group preferred. Both sides come from tracks with at least one annotation. One track can appear in several genre comparisons because FMA is multilabel. Use the full holdouts for primary metrics and the balanced view to examine per-class errors at equal support.

In [4]:
manifests = {}
coverage = {}
balanced = {}
for version in versions:
    directory = ready / version
    columns = [
        "track_id", "split", "artist_id", "album_id",
        "duration_seconds", "genre_ids", "duplicate_group",
    ]
    tracks = pd.read_parquet(directory / "tracks.parquet", columns=columns)
    manifests[version] = tracks.loc[tracks["split"].notna()].set_index("track_id")
    coverage[version] = {
        "Catalogue rows": len(tracks),
        "Assigned tracks": len(manifests[version]),
        "Ineligible rows": tracks["split"].isna().sum(),
        "Missing artist": tracks["artist_id"].isna().sum(),
        "Missing album": tracks["album_id"].isna().sum(),
        "Missing duration": tracks["duration_seconds"].isna().sum(),
        "No genre annotation": tracks["genre_ids"].map(len).eq(0).sum(),
    }
    if (directory / "balanced_eval.parquet").is_file():
        view = pd.read_parquet(directory / "balanced_eval.parquet")
        for split, rows in view.groupby("split"):
            counts = rows.groupby(["genre_id", "label_present"]).size()
            assert counts.nunique() == 1
            balanced[version, split] = {
                "Scored genres": rows["genre_id"].nunique(),
                "Tracks per genre per side": counts.iloc[0],
                "Track–genre rows": len(rows),
            }
    else:
        print(f"{version}: balanced evaluation file unavailable.")
display(pd.DataFrame(coverage).T)
if balanced:
    display(pd.DataFrame(balanced).T.rename_axis(["Version", "Split"]))

,Catalogue rows,Assigned tracks,Ineligible rows,Missing artist,Missing album,Missing duration,No genre annotation
pre-audit-v1,106574,106574,0,0,1025,16,2231
pre-audit-rare-support-v1,106574,106574,0,0,1025,16,2231


Scored genres  \
Version                   Split                       
pre-audit-v1              test                   16   
                          validation             16   
pre-audit-rare-support-v1 test                   16   
                          validation             16   

                                      Tracks per genre per side  \
Version                   Split                                   
pre-audit-v1              test                               71   
                          validation                         71   
pre-audit-rare-support-v1 test                              149   
                          validation                        149   

                                      Track–genre rows  
Version                   Split                         
pre-audit-v1              test                    2272  
                          validation              2272  
pre-audit-rare-support-v1 test                    4768  
                          validation              4768

## Follow-up to E1-04

E1-04 found that sparse genres can be dominated by a few artists and that duration cutoffs affect genres differently. Compare the same track IDs under the published assignment and the new splits. The label check compares the manifest's ancestor expansion with FMA's published `genres_all`, which E1-04 used for its inclusive counts. [E1-04 metadata exploration](https://github.com/piotrczech/music-atlas/blob/master/notebooks/fma_metadata.ipynb)

The grouped allocator keeps artists and duplicate groups within a split. A release with multiple artists can still span splits, so the album count measures a separate source of dependence.

The last tables follow the five least common top-level genres and the three most affected by a 10-minute cutoff in the selected catalogue. Read the largest artist's share alongside the number of tracks and artists to see whether extra tracks diversify the evaluation. The duration columns reproduce the E1-04 cutoffs using published durations and show what would be lost in each partition.

In [5]:
from ast import literal_eval

if manifests:
    metadata = data / "fma/fma_metadata"
    fields = {
        ("track", "genres_all"): "published_genres",
        ("track", "duration"): "published_duration",
        ("set", "split"): "published_split",
    }
    original = pd.read_csv(
        metadata / "tracks.csv", index_col=0, header=[0, 1], low_memory=False
    )[list(fields)].copy()
    original.columns = list(fields.values())
    original["published_genres"] = (
        original["published_genres"].fillna("[]").map(literal_eval).map(frozenset)
    )
    original["published_split"] = original["published_split"].replace("training", "train")
    genres = pd.read_csv(metadata / "genres.csv", index_col="genre_id")
    roots = genres.loc[genres["parent"].eq(0), "title"]
    root_ids = frozenset(roots.index)

    agreement = {}
    leakage = {}
    branch_profiles = {}
    for version, assigned in manifests.items():
        rows = assigned.join(original, validate="one_to_one")
        reference_tags = rows["published_genres"].map(
            lambda tags: tags if isinstance(tags, frozenset) else frozenset()
        )
        manifest_tags = rows["genre_ids"].map(frozenset)
        agreement[version] = {
            "Tracks compared": len(rows),
            "Missing FMA metadata rows": len(rows.index.difference(original.index)),
            "Different expanded tags": manifest_tags.ne(reference_tags).sum(),
            "Different top-level tags": (
                manifest_tags.map(root_ids.intersection)
                .ne(reference_tags.map(root_ids.intersection)).sum()
            ),
        }

        assignments = {"Published FMA": "published_split", "Versioned": "split"}
        for assignment, split_column in assignments.items():
            leakage[version, assignment] = {
                label: rows.groupby(field)[split_column].nunique().gt(1).sum()
                for field, label in {
                    "artist_id": "Artists", "album_id": "Albums",
                    "duplicate_group": "Duplicate groups",
                }.items()
            }

        duration = rows["published_duration"]
        branches = rows.assign(
            genre_id=reference_tags,
            short=duration.gt(0) & duration.lt(10),
            long=duration.gt(600) & duration.lt(float("inf")),
        ).explode("genre_id")
        branch_profiles[version] = branches.loc[
            branches["genre_id"].isin(roots.index),
            ["split", "genre_id", "artist_id", "short", "long"],
        ]

    display(pd.DataFrame(agreement).T)
    display(pd.DataFrame(leakage).T.rename_axis(["Version", "Assignment"]))

,Tracks compared,Missing FMA metadata rows,Different expanded tags,Different top-level tags
pre-audit-v1,106574,0,0,0
pre-audit-rare-support-v1,106574,0,0,0


Artists  Albums  Duplicate groups
Version                   Assignment                                      
pre-audit-v1              Published FMA        0     670                11
                          Versioned            0     777                 0
pre-audit-rare-support-v1 Published FMA        0     670                11
                          Versioned            0     784                 0

In [6]:
def branch_profile(rows):
    keys = ["split", "genre_id"]
    stats = rows.groupby(keys).agg(
        tracks=("artist_id", "size"),
        artists=("artist_id", "nunique"),
        short=("short", "mean"),
        long=("long", "mean"),
    )
    top_artist = rows.groupby([*keys, "artist_id"]).size().groupby(level=[0, 1]).max()
    stats["top_artist"] = top_artist / stats["tracks"]
    return stats


if manifests:
    for version, branches in branch_profiles.items():
        rows = pd.concat([branches.assign(split="Catalogue"), branches])
        stats = branch_profile(rows)
        catalogue = stats.loc["Catalogue"]
        focus = catalogue.nsmallest(5, "tracks").index.union(
            catalogue.nlargest(3, "long").index, sort=False
        )
        profile = stats.loc[(slice(None), focus), :].reset_index()
        profile["Genre"] = profile["genre_id"].map(roots)
        profile[["short", "long", "top_artist"]] *= 100
        profile = profile.rename(columns={
            "short": "Under 10 s (%)", "long": "Over 10 min (%)",
            "top_artist": "Top artist (%)", "tracks": "Tracks", "artists": "Artists",
        })
        print(version)
        display(profile.drop(columns="genre_id").set_index(["Genre", "split"])
                .reindex(pd.MultiIndex.from_product(
                    [roots.loc[focus], ["Catalogue", "train", "validation", "test"]],
                    names=["Genre", "split"],
                )).round(1))

pre-audit-v1


Tracks  Artists  Under 10 s (%)  \
Genre               split                                         
Easy Listening      Catalogue      730      170             0.0   
                    train          588      134             0.0   
                    validation      71       18             0.0   
                    test            71       18             0.0   
Old-Time / Historic Catalogue      868      520             0.8   
                    train          699      407             1.0   
                    validation      85       56             0.0   
                    test            84       57             0.0   
Soul-RnB            Catalogue     1499      265             0.1   
                    train         1203      211             0.1   
                    validation     147       27             0.0   
                    test           149       27             0.0   
Blues               Catalogue     1752      169             0.1   
                    train         1410      135             0.1   
                    validation     171       17             0.0   
                    test           171       17             0.0   
Spoken              Catalogue     1876      379             0.4   
                    train         1502      302             0.3   
                    validation     185       39             0.0   
                    test           189       38             1.6   
Jazz                Catalogue     4126      711             0.1   
                    train         3306      569             0.1   
                    validation     411       71             0.0   
                    test           409       71             0.0   
Experimental        Catalogue    38154     6949             0.3   
                    train        30532     5558             0.3   
                    validation    3810      696             0.4   
                    test          3812      695             0.3   

                                Over 10 min (%)  Top artist (%)  
Genre               split                                        
Easy Listening      Catalogue               1.2            10.4  
                    train                   0.9            12.9  
                    validation              5.6            19.7  
                    test                    0.0            14.1  
Old-Time / Historic Catalogue               1.3             3.0  
                    train                   1.4             3.7  
                    validation              1.2             8.2  
                    test                    0.0             9.5  
Soul-RnB            Catalogue               2.8             5.4  
                    train                   2.7             6.7  
                    validation              3.4            15.6  
                    test                    2.7            20.8  
Blues               Catalogue               2.1            20.0  
                    train                   1.9            24.8  
                    validation              2.9            24.0  
                    test                    2.9            24.0  
Spoken              Catalogue              14.7             7.4  
                    train                  16.2             9.2  
                    validation             11.4            22.7  
                    test                    5.8            46.0  
Jazz                Catalogue              12.9             5.3  
                    train                  12.3             6.6  
                    validation             10.0             8.3  
                    test                   20.0            23.5  
Experimental        Catalogue              10.1             1.9  
                    train                  10.0             2.4  
                    validation              9.6             4.4  
                    test                   11.4             3.4

pre-audit-rare-support-v1


Tracks  Artists  Under 10 s (%)  \
Genre               split                                         
Easy Listening      Catalogue      730      170             0.0   
                    train          432      104             0.0   
                    validation     149       33             0.0   
                    test           149       33             0.0   
Old-Time / Historic Catalogue      868      520             0.8   
                    train          512      327             0.0   
                    validation     178       96             3.9   
                    test           178       97             0.0   
Soul-RnB            Catalogue     1499      265             0.1   
                    train          998      177             0.1   
                    validation     253       44             0.0   
                    test           248       44             0.0   
Blues               Catalogue     1752      169             0.1   
                    train         1246      121             0.1   
                    validation     260       24             0.0   
                    test           246       24             0.0   
Spoken              Catalogue     1876      379             0.4   
                    train         1375      279             0.3   
                    validation     250       50             0.4   
                    test           251       50             1.2   
Jazz                Catalogue     4126      711             0.1   
                    train         3307      568             0.1   
                    validation     408       72             0.0   
                    test           411       71             0.0   
Experimental        Catalogue    38154     6949             0.3   
                    train        30532     5558             0.3   
                    validation    3810      696             0.4   
                    test          3812      695             0.2   

                                Over 10 min (%)  Top artist (%)  
Genre               split                                        
Easy Listening      Catalogue               1.2            10.4  
                    train                   1.4             8.1  
                    validation              0.7            51.0  
                    test                    1.3            14.8  
Old-Time / Historic Catalogue               1.3             3.0  
                    train                   1.0             4.5  
                    validation              2.2            14.6  
                    test                    1.1            13.5  
Soul-RnB            Catalogue               2.8             5.4  
                    train                   3.3             5.8  
                    validation              2.8            16.6  
                    test                    0.8            32.7  
Blues               Catalogue               2.1            20.0  
                    train                   2.5            28.1  
                    validation              0.4            87.7  
                    test                    2.0            24.8  
Spoken              Catalogue              14.7             7.4  
                    train                  12.2            10.0  
                    validation             27.2            18.4  
                    test                   15.5            34.7  
Jazz                Catalogue              12.9             5.3  
                    train                  13.1             6.6  
                    validation             10.0             7.6  
                    test                   14.1            24.3  
Experimental        Catalogue              10.1             1.9  
                    train                  10.1             2.4  
                    validation             11.3             3.1  
                    test                    9.2             3.4